In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

catalog_path = "/Users/nityaarya/Downloads/Worth-the-Watch/Data/Processed/combined_streaming_catalog_with_originals.csv"
catalog_df = pd.read_csv(catalog_path)

pricing_data = {
    'Platform': ['Netflix', 'Amazon Prime Video', 'Hulu', 'Max', 'Apple TV+'],
    'Plan': ['Standard (Ad-Free)', 'Prime with Video', 'No Ads', 'Ad-Free', 'Standard'],
    'Monthly Price': [17.99, 17.99, 18.99, 16.99, 9.99]
}
pricing_df = pd.DataFrame(pricing_data)

summary = catalog_df.groupby('platform').agg(
    Total_Titles=('title', 'count'),
    Total_Originals=('is_original', lambda x: (x == 'Y').sum()),
    Avg_IMDb_Rating=('imdbAverageRating', 'mean'),
    Weighted_IMDb_Rating=('imdbAverageRating', lambda x: np.average(
        x.fillna(0), weights=catalog_df.loc[x.index, 'imdbNumVotes'].fillna(0))),
    Genre_Variety=('genres', lambda x: len(set(g for sublist in x.dropna().apply(eval) for g in sublist)))
).reset_index()

platform_name_map = {
    'netflix': 'Netflix',
    'prime': 'Amazon Prime Video',
    'hulu': 'Hulu',
    'hbo_max': 'Max',
    'apple_tv': 'Apple TV+'
}
summary['Platform'] = summary['platform'].map(platform_name_map)

summary = summary.merge(pricing_df, on='Platform', how='left')

summary['Titles_per_Dollar'] = summary['Total_Titles'] / summary['Monthly Price']
summary['Originals_per_Dollar'] = summary['Total_Originals'] / summary['Monthly Price']
summary['Weighted_IMDb_per_Dollar'] = summary['Weighted_IMDb_Rating'] / summary['Monthly Price']

scaler = MinMaxScaler()
summary[['Norm_Titles_per_Dollar', 'Norm_Weighted_IMDb_per_Dollar', 'Norm_Originals_per_Dollar']] = scaler.fit_transform(
    summary[['Titles_per_Dollar', 'Weighted_IMDb_per_Dollar', 'Originals_per_Dollar']]
)

summary['Value_Index'] = (
    summary['Norm_Titles_per_Dollar'] +
    summary['Norm_Weighted_IMDb_per_Dollar'] +
    summary['Norm_Originals_per_Dollar']
)

summary_sorted = summary.sort_values(by='Value_Index', ascending=False)

summary_sorted[['Platform', 'Monthly Price', 'Total_Titles', 'Total_Originals',
                'Avg_IMDb_Rating', 'Weighted_IMDb_Rating', 'Genre_Variety',
                'Titles_per_Dollar', 'Originals_per_Dollar', 'Weighted_IMDb_per_Dollar', 
                'Norm_Titles_per_Dollar', 'Norm_Weighted_IMDb_per_Dollar', 'Norm_Originals_per_Dollar',
                'Value_Index']]


,Platform,Monthly Price,Total_Titles,Total_Originals,Avg_IMDb_Rating,Weighted_IMDb_Rating,Genre_Variety,Titles_per_Dollar,Originals_per_Dollar,Weighted_IMDb_per_Dollar,Norm_Titles_per_Dollar,Norm_Weighted_IMDb_per_Dollar,Norm_Originals_per_Dollar,Value_Index
0,Apple TV+,9.99,16928,208,6.367966,7.200682,31,1694.494494,20.820821,0.720789,0.379893,1.000000,0.092525,1.472418
3,Netflix,17.99,20084,1967,6.400291,7.258466,29,1116.397999,109.338521,0.403472,0.197540,0.025520,1.000000,1.223061
4,Amazon Prime Video,17.99,65850,443,5.952789,7.171655,34,3660.366870,24.624792,0.398647,1.000000,0.010701,0.131523,1.142224
1,Max,16.99,9166,960,6.693769,7.388401,31,539.493820,56.503826,0.434868,0.015564,0.121935,0.458344,0.595843
2,Hulu,18.99,9308,224,6.574679,7.504130,31,490.152712,11.795682,0.395162,0.000000,0.000000,0.000000,0.000000


In [3]:
#summary_sorted.to_csv("/Users/nityaarya/Downloads/Worth-the-Watch/Data/Processed/platform_value_analysis.csv", index=False)
#print("CSV file saved to: /Users/nityaarya/Downloads/Worth-the-Watch/Data/Processed/platform_value_analysis.csv")


CSV file saved to: /Users/nityaarya/Downloads/Worth-the-Watch/Data/Processed/platform_value_analysis.csv
